In [ ]:
# Structure features in token

from nupack import *
import RNA
import math
import itertools

def DNA_to_RNA(DNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'T': 'U'}
    return ''.join(match.get(base, base) for base in (DNA))
            
def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

def get_segments_interrupted_by_plus(input_string: str) -> list:
    # Split the string by the "+" character
    segments = input_string.split("+")
    
    # Remove empty segments if any (e.g., if the string starts or ends with "+")
    segments = [segment for segment in segments if segment]
    
    return segments

def structure_to_token(char):
    mapping = {
        ".": 0,
        "(": 1,
        ")": 2
    }
    return mapping.get(char, 3)

file_paths = [
    'Feature_CNN2_token.txt'
]

for file_path in file_paths:
    with open(file_path, 'w') as file:
        pass  

# Define NUPACK model-same as CRISPR trans-cleavage reaction
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

# Load crRNA sequences
file1 = open('RfxCas13d_mmc4_filtered_full_guide_sequences_NM_002794_4.txt', 'r')    
lines = file1.readlines()
guide_array = [line.strip() for line in lines]

# Load target sequences
file2 = open('RfxCas13d_mmc4_filtered_target_sequences_noPAM_NM_002794_4.txt', 'r')    
lines = file2.readlines()
full_target_array = [line.strip() for line in lines]
file3 = open('RfxCas13d_mmc4_filtered_target_sequences_noPAM_NM_002794_4.txt', 'r')    
lines = file3.readlines()
truncated_target_array = [line.strip() for line in lines]

file4 = open('filtered_matched_positions_NM_002794_4.txt', 'r')    
lines = file4.readlines()
start_position_array = [line.strip() for line in lines]

file5 = open('NM_002794.4.txt', 'r')
complete_target_part1 = file5.readlines()
complete_target_part1 = complete_target_part1[0]

subopt_structures_target_bh_part1 = subopt(strands=complete_target_part1, energy_gap=0.1, model=my_model_RNA)

for i in range (0, len(guide_array)):

     # Initialize struct_array
    struct_array = []
    struct_array_token = []

    guide = guide_array[i]
    target_full = full_target_array[i]
    target_truncated = truncated_target_array[i]

    # Compute suboptimal structures and energy
    subopt_structures_guide_bh = subopt(strands=guide, energy_gap=3, model=my_model_RNA)  
    subopt_structures_hybrid_ah = subopt(strands=[guide, target_truncated], energy_gap=3, model=my_model_RNA)

    if i < 2177:
        
        # Guide structure before hybridization
        for j in range (0, 66):
            
            try:
                to_be_added = structure_to_token(str(subopt_structures_guide_bh[0].structure)[j])
                    
            except IndexError:
                to_be_added = 3

            struct_array_token.append(to_be_added)
            
        # Target structure before hybridization 
        for j in range (int(start_position_array[i])-1, int(start_position_array[i]) + 29):
            
            try:
                to_be_added = structure_to_token(str(subopt_structures_target_bh_part1[0].structure)[j])
                    
            except IndexError:
                to_be_added = 3
    
            struct_array_token.append(to_be_added)

         
        # Hybrid structure after hybridization 
        hybrid_ah_1 = get_segments_interrupted_by_plus(str(subopt_structures_hybrid_ah[0].structure))[0]
        hybrid_ah_2 = get_segments_interrupted_by_plus(str(subopt_structures_hybrid_ah[0].structure))[1]
    
        for j in range (0, 66):
            
            try:
                to_be_added = structure_to_token(hybrid_ah_1[j])
                
            except IndexError:
                to_be_added = 3
            
            struct_array_token.append(to_be_added)

     
        for j in range (0, 30):
            
            try:
                to_be_added = structure_to_token(hybrid_ah_2[j])
                
            except IndexError:
                to_be_added = 3
            
            struct_array_token.append(to_be_added)
            
    struct_array_to_append = [struct_array_token]
    struct_array.append(struct_array_to_append)

    struct_array = list(itertools.chain.from_iterable(struct_array))
    
    # Open a file in write mode
    with open('Feature_CNN2_token.txt', 'a') as file:
        # Iterate over each row in the 2D array
        for row in struct_array:
            # Convert each element to a string and join them with spaces
            file.write(' '.join(map(str, row)) + '\n')
        file.write('\n')

print('-------------')